# 📊 Eksplorasi Data & Analisis Sentimen Ulasan Produk (EDA)
**Mata Kuliah:** Kecerdasan Buatan — Politeknik Negeri Indramayu  
**Dataset:** PRDECT-ID (Tokopedia) + Hugging Face Indonesian E-Commerce  
**Algoritma:** Naive Bayes Classifier (MultinomialNB & ComplementNB)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import sys
from pathlib import Path

# Setup visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

sys.path.insert(0, '..')
from src.preprocess import DataPreprocessor


## 1. Memuat Dataset Ulasan Produk
Dataset memuat ulasan produk berbahasa Indonesia yang telah dipetakan menjadi 3 kelas sentimen: **positif**, **netral**, dan **negatif**.

In [ ]:
df = pd.read_csv('../data/reviews.csv')
print(f'Total baris dataset: {len(df):,} ulasan')
df.head()


## 2. Analisis Distribusi Kelas Sentimen
Melihat keseimbangan antara kelas positif, netral, dan negatif.

In [ ]:
class_counts = df['sentiment'].value_counts()
print(class_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = {'positif': '#10b981', 'netral': '#f59e0b', 'negatif': '#f43f5e'}
sns.barplot(x=class_counts.index, y=class_counts.values, palette=colors, ax=axes[0])
axes[0].set_title('Frekuensi per Kelas Sentimen')
axes[0].set_ylabel('Jumlah Ulasan')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
            colors=[colors[k] for k in class_counts.index], startangle=140)
axes[1].set_title('Proporsi Sentimen Ulasan')

plt.tight_layout()
plt.show()


## 3. Distribusi Panjang Karakter dan Kata
Analisis apakah panjang ulasan berkorelasi dengan sentimen tertentu.

In [ ]:
df['char_length'] = df['text'].astype(str).apply(len)
df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='sentiment', y='char_length', palette=colors, ax=axes[0])
axes[0].set_title('Distribusi Panjang Karakter per Sentimen')
axes[0].set_ylabel('Jumlah Karakter')

sns.boxplot(data=df, x='sentiment', y='word_count', palette=colors, ax=axes[1])
axes[1].set_title('Distribusi Jumlah Kata per Sentimen')
axes[1].set_ylabel('Jumlah Kata')

plt.tight_layout()
plt.show()


## 4. Evaluasi Pipeline Preprocessing Teks
Membandingkan teks ulasan asli dengan hasil pembersihan, normalisasi kata informal (*slang*), dan penyaringan stopword.

In [ ]:
p = DataPreprocessor()
sample_texts = [
    'Barang original 100%, pengemasan super rapi dan pengiriman cepet bgt! Recommended!',
    'Lumayan, tp jangan expect banyak lah. Kualitas standar bgt sesuai harga.',
    'Brg rusak parah pas sampe, paking ancur dan seller ga tanggung jawab! Kecewa!'
]

for s in sample_texts:
    cleaned = p.preprocess_text(s)
    print(f'Original: {s}')
    print(f'Cleaned : {cleaned}
')


## 5. Visualisasi Kata Kunci (WordCloud) per Kelas

In [ ]:
# Siapkan data bersih untuk wordcloud
df['cleaned'] = df['text'].astype(str).apply(p.preprocess_text)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, sentiment in enumerate(['positif', 'netral', 'negatif']):
    text_corpus = ' '.join(df[df['sentiment'] == sentiment]['cleaned'])
    wc = WordCloud(width=400, height=300, background_color='white',
                   colormap='Greens' if sentiment=='positif' else ('YlOrBr' if sentiment=='netral' else 'Reds')).generate(text_corpus)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(f'WordCloud — Sentimen {sentiment.upper()}', fontsize=14)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()
